# Applied OOP: Deep Learning Modules, Experience Replay Buffers, and Autograd Tapes

**Session:** 2026-11-03 · OOP Practice Lab  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

---

## Overview

In this notebook, we connect OOP principles, bounded queues, and stacks to machine learning architectures:
1. **Layer Hierarchy & Sequential Container:** Building an extensible mini deep-learning framework with an abstract `Layer` base class, concrete `LinearLayer` and `ReLULayer` subclasses, and a composite `SequentialContainer`.
2. **Experience Replay Buffer:** Building a circular buffer for Reinforcement Learning (Q-learning) that overwrites the oldest experiences upon reaching capacity and provides uniform random batch sampling.
3. **Autograd Computation Tape Evaluator:** Leveraging a LIFO Stack to evaluate reverse-mode backward automatic differentiation graphs.


## 1. Modular Deep Learning Framework: Layer Subclassing

Modern frameworks like PyTorch structure neural networks via class inheritance and aggregation.


In [ ]:
import math
import random
from abc import ABC, abstractmethod


class Layer(ABC):
    """Abstract base class establishing the contract for neural network layers."""

    @abstractmethod
    def forward(self, x: list[float]) -> list[float]:
        """Compute the forward transformation over input activations."""
        pass

    @abstractmethod
    def parameter_count(self) -> int:
        """Return the number of trainable scalar parameters."""
        pass


class LinearLayer(Layer):
    """Fully connected affine transformation layer: y = W * x + b."""

    def __init__(self, in_features: int, out_features: int, seed: int = 42) -> None:
        self.in_features: int = in_features
        self.out_features: int = out_features
        rng = random.Random(seed)
        # Xavier uniform initialization scale
        scale = math.sqrt(2.0 / (in_features + out_features))
        self.weights: list[list[float]] = [
            [rng.uniform(-scale, scale) for _ in range(in_features)] for _ in range(out_features)
        ]
        self.biases: list[float] = [0.0] * out_features

    def forward(self, x: list[float]) -> list[float]:
        if len(x) != self.in_features:
            raise ValueError(f"Input dimension mismatch: expected {self.in_features}, got {len(x)}")
        outputs = []
        for i in range(self.out_features):
            dot = sum(self.weights[i][j] * x[j] for j in range(self.in_features))
            outputs.append(dot + self.biases[i])
        return outputs

    def parameter_count(self) -> int:
        return (self.in_features * self.out_features) + self.out_features


class ReLULayer(Layer):
    """Element-wise Rectified Linear Unit activation: f(x) = max(0, x)."""

    def forward(self, x: list[float]) -> list[float]:
        return [max(0.0, val) for val in x]

    def parameter_count(self) -> int:
        return 0


class SequentialContainer(Layer):
    """Composite container aggregating an ordered sequence of Layers."""

    def __init__(self, layers: list[Layer]) -> None:
        self.__layers: list[Layer] = []
        for layer in layers:
            self.add_layer(layer)

    def add_layer(self, layer: Layer) -> None:
        if not isinstance(layer, Layer):
            raise TypeError(f"Expected Layer instance, got {type(layer).__name__}")
        self.__layers.append(layer)

    def forward(self, x: list[float]) -> list[float]:
        activations = x
        for layer in self.__layers:
            activations = layer.forward(activations)
        return activations

    def parameter_count(self) -> int:
        return sum(layer.parameter_count() for layer in self.__layers)


# Test Multi-Layer Perceptron (MLP) construction
mlp = SequentialContainer(
    [
        LinearLayer(in_features=4, out_features=8, seed=1),
        ReLULayer(),
        LinearLayer(in_features=8, out_features=2, seed=2),
    ]
)

x_sample = [0.5, -1.2, 3.0, 0.1]
y_pred = mlp.forward(x_sample)
assert len(y_pred) == 2
# Params: (4*8 + 8) + 0 + (8*2 + 2) = 40 + 18 = 58
assert mlp.parameter_count() == 58
print(f"MLP forward output: {y_pred}, Total parameters: {mlp.parameter_count()}")

## 2. Deep Reinforcement Learning: Experience Replay Buffer

In Deep Q-Learning, an agent records environment interactions $e_t = (s_t, a_t, r_t, s_{t+1}, d_t)$. The replay buffer is a bounded circular queue: once full, incoming transitions overwrite the oldest memory in $O(1)$ time.


In [ ]:
from typing import NamedTuple


class Experience(NamedTuple):
    """Immutable transition tuple for reinforcement learning."""

    state: list[float]
    action: int
    reward: float
    next_state: list[float]
    done: bool


class ExperienceReplayBuffer:
    """Bounded circular memory buffer for RL experience storage and sampling."""

    def __init__(self, capacity: int, seed: int = 42) -> None:
        self.__capacity: int = int(capacity)
        self.__storage: list[Experience | None] = [None] * self.__capacity
        self.__next_idx: int = 0
        self.__size: int = 0
        self.__rng: random.Random = random.Random(seed)

    def add(self, exp: Experience) -> None:
        """Append transition, overwriting oldest entry when capacity is reached."""
        self.__storage[self.__next_idx] = exp
        self.__next_idx = (self.__next_idx + 1) % self.__capacity
        self.__size = min(self.__size + 1, self.__capacity)

    def sample_batch(self, batch_size: int) -> list[Experience]:
        """Sample a uniform random batch of experiences from valid memory."""
        if batch_size > self.__size:
            raise ValueError(
                f"Requested batch_size {batch_size} exceeds current buffer size {self.__size}"
            )
        valid_pool = [e for e in self.__storage if e is not None]
        return self.__rng.sample(valid_pool, batch_size)

    def __len__(self) -> int:
        return self.__size


# Verification: Buffer wrapping and sampling
buffer = ExperienceReplayBuffer(capacity=5, seed=123)
for i in range(8):
    exp = Experience(
        state=[float(i)], action=i % 2, reward=float(i * 10), next_state=[float(i + 1)], done=False
    )
    buffer.add(exp)

assert len(buffer) == 5  # Strictly capped at capacity
batch = buffer.sample_batch(batch_size=3)
assert len(batch) == 3
print("Replay buffer capacity cap and sampling verified successfully.")

## 3. Autograd Reverse-Mode Differentiation Tape via Stack

Reverse-mode automatic differentiation unwinds a forward execution tape in reverse order using a LIFO Stack to compute parameter gradients.


In [ ]:
from typing import Any, NamedTuple


class BackwardOp(NamedTuple):
    name: str
    vjp_fn: Any  # Vector-Jacobian Product function


class AutogradTape:
    """Execution tape using a Stack to evaluate backward automatic differentiation."""

    def __init__(self, max_ops: int = 100) -> None:
        self.__tape: list[BackwardOp] = []

    def record_operation(self, name: str, vjp_fn: Any) -> None:
        """Push forward operation's VJP onto the backward tape stack."""
        self.__tape.append(BackwardOp(name, vjp_fn))

    def backward(self, initial_gradient: float = 1.0) -> list[tuple[str, float]]:
        """Unwind the execution tape in LIFO reverse order."""
        grad = initial_gradient
        execution_log = []
        while self.__tape:
            op = self.__tape.pop()  # LIFO Stack pop
            grad = op.vjp_fn(grad)
            execution_log.append((op.name, grad))
        return execution_log


# Example: y = ((x * 2) + 3)^2 evaluated at x = 4
# Forward:
#   v1 = x * 2  = 8
#   v2 = v1 + 3 = 11
#   y  = v2 ^ 2 = 121
# Backward VJPs:
#   dy/dv2 = 2 * v2 = 22
#   dv2/dv1 = 1.0
#   dv1/dx  = 2.0
#   Total dy/dx = 22 * 1.0 * 2.0 = 44.0

tape = AutogradTape()
x = 4.0
v1 = x * 2.0
tape.record_operation("mul_2", lambda g: g * 2.0)
v2 = v1 + 3.0
tape.record_operation("add_3", lambda g: g * 1.0)
y = v2**2
tape.record_operation("square_v2", lambda g: g * (2.0 * v2))

grad_trace = tape.backward(initial_gradient=1.0)
final_grad = grad_trace[-1][1]
print("Backward execution trace:", grad_trace)
print("Computed dy/dx:", final_grad)
assert final_grad == 44.0
print(
    "Autograd Stack tape verified: dy/dx = 44.0 matches analytical derivative d/dx[(2x+3)^2] at x=4."
)